In [36]:
import random
import numpy as np

In [43]:
class LinearGridworld:
    """
    Generic 1-D Linear Gridworld Environment.
    
    Default setup for Exercise 5:
    - 5 states: State 0 (Trap, -10), State 1 (Start), States 2 &amp; 3, State 4 (Goal, +10)
    - Actions: 0 = Left (s - 1), 1 = Right (s + 1)
    """
    def __init__(self, num_states=5, terminal_rewards={0: -10.0, 4: 10.0}, step_reward=0.0):
        self.num_states = num_states
        self.num_actions = 2  # 0: Left, 1: Right
        self.terminal_rewards = terminal_rewards
        self.step_reward = step_reward
        self.current_state = 1  # Default start state

    def reset(self, start_state=None):
        """
        Resets environment to a start state. If start_state is None,
        randomly selects among intermediate non-terminal states (1, 2, or 3).
        """
        if start_state is None:
            non_terminals = [s for s in range(self.num_states) if s not in self.terminal_rewards]
            self.current_state = random.choice(non_terminals)
        else:
            self.current_state = start_state
        return self.current_state

    def step(self, action):
        """
        Executes action (0=Left, 1=Right) from current state.
        Returns: next_state, reward, is_terminal
        """
        step_dir = -1 if action == 0 else 1
        next_state = self.current_state + step_dir
        next_state = max(0, min(self.num_states - 1, next_state))
        self.current_state = next_state
        if next_state in self.terminal_rewards:
            reward = self.terminal_rewards[next_state]
            is_terminal = True
        else:
            reward = self.step_reward
            is_terminal = False

        return next_state, reward, is_terminal

In [44]:
# Hyperparameters
gamma = 0.9
alpha = 0.1
num_episodes = 2000
epsilon = 0.1

In [45]:
# Initialize Environment
env = LinearGridworld(num_states=5, terminal_rewards={0: -10.0, 4: 10.0}, step_reward=0.0)
q_table = np.zeros((env.num_states, env.num_actions))

In [46]:
# Training Loop
for episode in range(num_episodes):
    state = env.reset()
    action = random.choice([0, 1])
    is_terminal = False
    first_step = True

    while not is_terminal:
        if not first_step:
            if random.random() < epsilon:
                action = random.choice([0, 1])
            else:
                action = np.argmax(q_table[state])
        
        first_step = False
        next_state, reward, is_terminal = env.step(action)
        best_next_q = np.max(q_table[next_state]) if not is_terminal else 0.0
        q_table[state, action] += alpha * (reward + gamma * best_next_q - q_table[state, action])
        state = next_state

In [47]:
# Print Final Converged Q-Table
print("Final Converged Q-Table")
print("State | Left (Action 0) | Right (Action 1)")
for s in range(env.num_states):
    print(f"  {s}   |    {q_table[s, 0]:8.4f}     |    {q_table[s, 1]:8.4f}")

Final Converged Q-Table
State | Left (Action 0) | Right (Action 1)
  0   |      0.0000     |      0.0000
  1   |    -10.0000     |      8.1000
  2   |      7.2900     |      9.0000
  3   |      8.1000     |     10.0000
  4   |      0.0000     |      0.0000


In [48]:
# Test Optimal Policy
print("Testing Optimal Policy Trajectories")
for start_pos in [7-9]:
    state = env.reset(start_state=start_pos)
    path = [state]
    is_terminal = False
    
    while not is_terminal:
        best_action = np.argmax(q_table[state])
        next_state, _, is_terminal = env.step(best_action)
        path.append(next_state)
        state = next_state
        
    print(f"Start State {start_pos} -> Path Traversed: {path}")

Testing Optimal Policy Trajectories
Start State -2 -> Path Traversed: [-2, 0]
